### Recruitment Agent

In [10]:
# Import the necessary libraries
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, input_guardrail, GuardrailFunctionOutput
from pydantic import BaseModel
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content

In [2]:
# Load environment variables from .env file
load_dotenv(override=True)  

True

In [3]:
# Agent workflow
formal_jd_instructions =  """You are a formal job description writer for a professional recruitment firm.
You write structured, corporate-style job descriptions with clearly defined sections:
Job Summary, Key Responsibilities, Required Qualifications, and Benefits.
Your tone is professional, precise and authoritative."""

engaging_jd_instructions = """You are a creative job description writer for modern, fast-growing startups.
You write warm, energetic and inclusive job descriptions that excite candidates.
Your tone is conversational, bold and human — focused on culture, impact and growth opportunities."""

concise_jd_instructions = """You are a concise job description writer who values brevity above all.
You write short, scannable job descriptions in under 150 words.
No fluff, no filler — just the role, key skills needed, and what's on offer."""

In [4]:
# Create three agents with different instructions and the same model
formal_jd_agent = Agent(
    name="Formal JD Agent",
    instructions=formal_jd_instructions,
    model="gpt-4o-mini"
)

engaging_jd_agent = Agent(
    name="Engaging JD Agent",
    instructions=engaging_jd_instructions,
    model="gpt-4o-mini"
)

concise_jd_agent = Agent(
    name="Concise JD Agent",
    instructions=concise_jd_instructions,
    model="gpt-4o-mini"
)

In [5]:
# Convert agents to tools with descriptions
description = "Generate a job description for a potential candidate"

tool1 = formal_jd_agent.as_tool(
    tool_name="formal_jd_generator",
    tool_description=description
)

tool2 = engaging_jd_agent.as_tool(
    tool_name="engaging_jd_generator",
    tool_description=description
)

tool3 = concise_jd_agent.as_tool(
    tool_name="concise_jd_generator",
    tool_description=description
)

tools = [tool1, tool2, tool3]

In [6]:
# Create two more agents for email subject line writing and HTML email formatting
subject_instructions = """You are an expert email subject line writer for a professional recruitment firm.
You are given a job description and you must write a single compelling subject line 
that grabs the board's attention and clearly communicates the role being hired for.
Return only the subject line — no explanation, no preamble."""

html_instructions = """You are an HTML email formatter for a professional recruitment firm.
You are given a job description in plain text or markdown and you must convert it 
into a clean, professional HTML email body with simple, clear layout and design.
Use proper HTML tags, headings, bullet points and spacing for readability.
Return only the HTML — no explanation, no markdown backticks."""


subject_writer_agent = Agent(
    name="Email Subject Writer",
    instructions=subject_instructions,
    model="gpt-4o-mini"
)

subject_tool = subject_writer_agent.as_tool(
    tool_name="subject_writer",
    tool_description="Write a subject line for the job description email"
)

html_converter_agent = Agent(
    name="HTML Email Converter",
    instructions=html_instructions,
    model="gpt-4o-mini"
)

html_tool = html_converter_agent.as_tool(
    tool_name="html_converter",
    tool_description="Convert the job description into an HTML email body"
)

In [11]:
# Function to send out the email using SendGrid
@function_tool
def send_html_email(subject:str, html_body:str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("iwanttotestanapp@gmail.com")
    to_email = To("ctrlplusstyle@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [12]:
# Combine all tools into a single list for the agent to use
ntools = [subject_tool, html_tool, send_html_email]

In [13]:
# Create the final agent that will manage the entire email creation and sending process
emailer_instructions = """You are an Email Manager at a professional recruitment firm.
You are given a finalized job description and you must do the following steps in order:

1. Use the subject_writer tool to generate a compelling email subject line for the job description.
2. Use the html_converter tool to convert the job description into a clean, professional HTML email body.
3. Use the send_email tool to send the email to the board with the subject and HTML body you generated.

Important rules:
- You must always run all three tools in order — never skip a step.
- Never write the subject line or HTML yourself — always use the tools.
- Only send the email once — never send duplicates."""

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=ntools,
    model="gpt-4o-mini"
)

In [14]:
# Define the tools and handoffs for the recruitment agent
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent] #Convert to Handoff

In [15]:
# Create the HR Manager agent who will generate the job descriptions and hand off to the Email Manager
hr_manager_instructions = """You are a Senior HR Manager at a professional recruitment firm.
Your goal is to find the single best job description to send to the board for approval.

Follow these steps carefully:

1. Generate Drafts: Use all three job description tools to generate three different drafts:
   - formal_jd_generator: for a structured, corporate-style job description
   - engaging_jd_generator: for a modern, startup-friendly job description
   - concise_jd_generator: for a short, to-the-point job description
   Do not proceed until all three drafts are generated.

2. Evaluate and Select: Review all three drafts and select the single best one based on:
   - Clarity and professionalism
   - Relevance to the role requested
   - Likelihood of attracting the right candidates
   You can re-run any tool if you are not satisfied with its output.

3. Handoff: Pass ONLY the winning job description to the Email Manager agent for sending.

Crucial Rules:
- You must use all three tools to generate drafts — never write the JD yourself.
- You must hand off exactly ONE job description to the Email Manager — never more than one.
- Never send the email yourself — that is the Email Manager's responsibility."""

hr_manager = Agent(
    name="HR Manager",
    instructions=hr_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
)

In [16]:
message = """We are looking to hire a Senior Full Stack Developer with expertise in 
React, Node.js, PostgreSQL and REST API development."""

with trace("Recruitment Agent"):
    result = await Runner.run(hr_manager, input=message)


print(result.final_output)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function c

The email has been successfully sent with the engaging job description for the Senior Full Stack Developer position. If you need any further assistance or follow-up actions, feel free to let me know!


#### Implementing Guardrail

In [17]:
# Define the output schema for the job description evaluation (if needed for guardrails or structured output)
class JobRequestOutput(BaseModel):
    has_job_title: bool
    has_required_skills: bool
    has_salary_range: bool
    salary_range: str
    

# Create a guardrail agent to validate the job request before processing
combined_guardrail_agent = Agent(
    name="Job Request Validator",
    instructions="""Analyze the job request and check for three things:
    1. Does it include a job title?
    2. Does it include required skills?
    3. Does it include a salary range? A salary range means a minimum and maximum 
       figure e.g '$50,000 - $70,000' or '80k - 100k'. 
       Vague terms like 'competitive salary' or 'up to 50,000' do NOT count as a range.
    
    Return your findings for all three checks.""",
    output_type=JobRequestOutput,
    model="gpt-4o-mini"
)

In [18]:
# Use the guardrail agent to validate the input before it reaches the HR Manager agent
@input_guardrail
async def guardrail_against_incomplete_request(ctx, agent, message):
    result = await Runner.run(
        combined_guardrail_agent, 
        input=message, 
        context=ctx.context
    )
    
    output = result.final_output
    
    # Tripwire fires if ANY of the three checks fail
    tripwire = (
        not output.has_job_title or
        not output.has_required_skills or
        not output.has_salary_range
    )
    
    return GuardrailFunctionOutput(
        output_info=output,
        tripwire_triggered=tripwire
    )

In [19]:
# Re-create the HR Manager agent with the guardrail in place
hr_manager = Agent(
    name="HR Manager",
    instructions=hr_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_incomplete_request]
)

In [20]:
# Test the entire workflow with a sample job request
message = """We are looking to hire a Senior Full Stack Developer with expertise in 
React, Node.js, PostgreSQL and REST API development."""

with trace("Recruitment Agent"):
    result = await Runner.run(hr_manager, input=message)


print(result.final_output)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

In [22]:
# Now test with a complete job request that should trigger the guardrail
message = """Hire a Senior Python Developer with Django, 
REST API and PostgreSQL skills, salary range $80,000 - $100,000"""

with trace("Recruitment Agent"):
    result = await Runner.run(hr_manager, input=message)


print(result.final_output)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


The email has been successfully sent with the engaging job description for the Senior Python Developer position. If there's anything else you need, feel free to ask!
